#### Environment Check

In [1]:
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from tqdm.auto import tqdm

#### Setup

In [2]:
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from tqdm.auto import tqdm

#### Project Paths

In [3]:
LAB_DIR = Path("..").resolve()
CODE_DIR = LAB_DIR / "code"
DATA_DIR = LAB_DIR / "data"

DATA_DIR.mkdir(exist_ok=True)

str(LAB_DIR), str(DATA_DIR)

('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab',
 '/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data')

#### Import Course Helpers

In [4]:
import sys

sys.path.append(str(CODE_DIR))

from ingest import load_faq_data
from evaluation_utils import (
    calc_total_price,
    llm_structured_retry,
    map_progress,
)

#### Load OpenAI Client

In [5]:
PROJECT_ROOT = LAB_DIR.parents[1]
ENV_PATH = PROJECT_ROOT / ".env"
loaded = load_dotenv(ENV_PATH)
print("Env file:", ENV_PATH)
print("Env loaded:", loaded)
openai_client = OpenAI()
MODEL = "gpt-5.4-mini"


#### Load FAQ Documents

In [6]:
documents = load_faq_data()

len(documents)

1375

#### Filter LLM Zoomcamp Documents

In [7]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm

len(documents)

113

#### Inspect One Document

In [8]:
doc = documents[0]

doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

#### Define Structured Output

In [9]:
class Questions(BaseModel):
    questions: list[str]

#### Ground Truth Generation Instructions

In [10]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.

Formulate 5 questions this student might ask based on a provided FAQ record.
The record should contain the answer to the questions, and the questions should be complete and not too short.

If possible, use as few words as possible from the record.

The questions should be different from each other.

Return the result as a JSON object with one field called questions.
""".strip()

#### Test Prompt On One Document

In [11]:
doc = documents[0]

user_prompt = json.dumps(doc, indent=2)

print(user_prompt)

{
  "id": "74eb249bbf",
  "course": "llm-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "I just discovered the course. Can I still join?",
  "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
}


#### Generate Questions For One Document

In [12]:
result, usage = llm_structured_retry(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model=MODEL,
)

result.questions

['I found this course late. Is it still okay to enroll now?',
 'Can someone who discovers the class after it starts still take part?',
 'If I join after the course has begun, am I still eligible for a certificate?',
 'What deadline should I keep in mind if I want certification after joining late?',
 'Does late enrollment affect whether I can earn a course certificate, and what do I need to do?']

#### Check Cost For One Call

In [13]:
from evaluation_utils import calc_price

calc_price(usage)

8.895e-05

#### Convert One Result To Ground Truth Records

In [14]:
records = []

for question in result.questions:
    records.append({
        "question": question,
        "document": doc["id"],
    })

records

[{'question': 'I found this course late. Is it still okay to enroll now?',
  'document': '74eb249bbf'},
 {'question': 'Can someone who discovers the class after it starts still take part?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has begun, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What deadline should I keep in mind if I want certification after joining late?',
  'document': '74eb249bbf'},
 {'question': 'Does late enrollment affect whether I can earn a course certificate, and what do I need to do?',
  'document': '74eb249bbf'}]

#### Create Function For One Document

In [15]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc, indent=2)

    result, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model=MODEL,
    )

    records = []

    for question in result.questions:
        records.append({
            "question": question,
            "document": doc["id"],
        })

    return records, usage

#### Test First 3 Documents

In [16]:
ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

  0%|          | 0/3 [00:00<?, ?it/s]

15

#### Preview Test Ground Truth

In [17]:
ground_truth[:5]

[{'question': 'I found this course late — is it still possible to start and participate now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already begun, can I still earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What condition do I need to meet to get a certificate if I’m joining late?',
  'document': '74eb249bbf'},
 {'question': 'Does late enrollment prevent me from submitting the final project?',
  'document': '74eb249bbf'},
 {'question': 'Are there any deadlines I should know about if I want certification after discovering the course now?',
  'document': '74eb249bbf'}]

#### Check Test Cost

In [18]:
calc_total_price(usages)

0.00034229999999999997

#### Generate Ground Truth For All Documents

In [19]:
ground_truth = []
usages = []

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

565

#### Convert To DataFrame

In [20]:
df_ground_truth = pd.DataFrame(ground_truth)

df_ground_truth.head()

,question,document
0,What is the recommended way to begin the cours...,04919992b3
1,Which course resources should I open first if ...,04919992b3
2,"How do the lectures, notebooks, and homework u...",04919992b3
3,Where can I find the deadlines and submit my h...,04919992b3
4,"Can I start the course at any time, and does t...",04919992b3


#### Save Ground Truth CSV

In [21]:
output_path = DATA_DIR / "ground_truth-new.csv"

df_ground_truth.to_csv(output_path, index=False)

output_path

PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data/ground_truth-new.csv')

#### Calculate Total Cost

In [22]:
total_cost = calc_total_price(usages)

total_cost

0.01434705

#### Final Summary

In [23]:
print("Documents:", len(documents))
print("Ground truth questions:", len(ground_truth))
print("Total cost:", total_cost)
print("Saved to:", output_path)

Documents: 113
Ground truth questions: 565
Total cost: 0.01434705
Saved to: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data/ground_truth-new.csv
